## Quick start : Compare runs choose a model , and deploy it to REST API

in this quickstart you will:

- Run a hyperparameter sweep on a traning script
- Compare the results of the runs in the MLflow UI
- Choose the best run and register it as a model
- Deploy the model to REST API
- Build a container image for deployment to a cloud platform 

hyperopt - library that support hyperparameter in ann 

In [2]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import mlflow
from mlflow import log_metric, log_param, log_artifacts
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import mlflow
from mlflow.models import infer_signature


In [7]:
## load dataset
data = pd.read_csv('data\wine.csv',sep=';')


In [8]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [13]:
data.count()

fixed acidity           1599
volatile acidity        1599
citric acid             1599
residual sugar          1599
chlorides               1599
free sulfur dioxide     1599
total sulfur dioxide    1599
density                 1599
pH                      1599
sulphates               1599
alcohol                 1599
quality                 1599
dtype: int64

In [14]:
## split data into training validating and testing 
## quality - dependent variable need to be predicted
## rest of the columns are independent variables that are used to predict quality means they are input features
train,test=train_test_split(data,test_size=0.25,random_state=42)
train


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
582,11.7,0.490,0.49,2.2,0.083,5.0,15.0,1.00000,3.19,0.43,9.2,5
626,8.8,0.600,0.29,2.2,0.098,5.0,15.0,0.99880,3.36,0.49,9.1,5
1030,7.1,0.590,0.00,2.1,0.091,9.0,14.0,0.99488,3.42,0.55,11.5,7
620,8.3,0.540,0.24,3.4,0.076,16.0,112.0,0.99760,3.27,0.61,9.4,5
490,9.3,0.775,0.27,2.8,0.078,24.0,56.0,0.99840,3.31,0.67,10.6,6
...,...,...,...,...,...,...,...,...,...,...,...,...
1130,9.1,0.600,0.00,1.9,0.058,5.0,10.0,0.99770,3.18,0.63,10.4,6
1294,8.2,0.635,0.10,2.1,0.073,25.0,60.0,0.99638,3.29,0.75,10.9,6
860,7.2,0.620,0.06,2.7,0.077,15.0,85.0,0.99746,3.51,0.54,9.5,5
1459,7.9,0.200,0.35,1.7,0.054,7.0,15.0,0.99458,3.32,0.80,11.9,7


In [16]:
# Train dataset
train_x = train.drop('quality', axis=1).to_numpy()
train_y = train['quality'].to_numpy().ravel()

# Test dataset
test_x = test.drop('quality', axis=1).to_numpy()
test_y = test['quality'].to_numpy().ravel()

# Split train into train & validation
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.20, random_state=42
)

# MLflow signature
signature = infer_signature(train_x, train_y)


In [17]:
np.mean(train_x,axis=0)

array([ 8.29781022,  0.53131387,  0.2712513 ,  2.52591241,  0.08946403,
       15.88008342, 47.07351408,  0.99671879,  3.30930136,  0.6632951 ,
       10.42846715])

In [ ]:
## ANN Model

def train_model(params,epochs,train_x,train_y,valid_x,valid_y):

    ## Define the model
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential( 
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)
        ]
    )

### Compile the model
model.compile(optimizer=keras.optimizers.SGD(
    learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss='mean_squared_error',
    metrics=[keras.metrics.RootMeanSquaredError()]


    )
    ## Train the Ann model with Lr and momentum parameters
    with mlflow.start_run(nested=True):
        model.fi












In [18]:
import numpy as np
import mlflow
import mlflow.tensorflow
from tensorflow import keras
from hyperopt import STATUS_OK


## ANN Model training function
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, signature):

    # Compute normalization statistics
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    # Define the ANN model
    model = keras.Sequential(
        [
            keras.Input(shape=(train_x.shape[1],)),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1)
        ]
    )

    # Compile the model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],
            momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # Train the ANN model with MLflow tracking
    with mlflow.start_run(nested=True):

        # Log hyperparameters
        mlflow.log_param("learning_rate", params["lr"])
        mlflow.log_param("momentum", params["momentum"])
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", 64)

        # Train model
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
            verbose=1
        )

        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64, verbose=0)
        eval_rmse = eval_result[1]

        # Log metric
        mlflow.log_metric("rmse", eval_rmse)

        # Log the model
        mlflow.tensorflow.log_model(
            model,
            artifact_path="model",
            signature=signature
        )

    return {
        "loss": eval_rmse,
        "status": STATUS_OK,
        "model": model
    }


In [19]:
def objective(params):
    result = train_model(
        params,
        epochs=50,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        signature=signature
    )
    return result 


In [20]:
space = {
    "lr" :hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 0.1)
}

In [21]:
mlflow.set_experiment("DL-MLflow-Wine-Quality")
with mlflow.start_run():
    ## Conduct hyperparameter optimization
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Fetch the best model
    best_run = sorted(trials.results, key=lambda x: x['loss'])[0]

    # Log the best parameters, loss , model
    mlflow.log_param(best)
    mlflow.log_metric("eval_rmse", best_run['loss'])
    mlflow.tensorflow.log_model(best_run['model'], "model", signature=signature)

    # print best hyperparameters and corresponding loss
    print(f"Best Parameters: {best}")
    print(f"Best eval RMSE: {best_run['loss']}")

2025/12/31 12:31:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/31 12:31:21 INFO mlflow.store.db.utils: Updating database tables
2025/12/31 12:31:21 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/31 12:31:21 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/31 12:31:21 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/31 12:31:21 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/31 12:31:21 INFO mlflow.tracking.fluent: Experiment with name 'DL-MLflow-Wine-Quality' does not exist. Creating a new experiment.


Epoch 1/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 12s 859ms/step - loss: 38.3812 - root_mean_squared_error: 6.1953
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 9.5744 - root_mean_squared_error: 3.0943 - val_loss: 1.8006 - val_root_mean_squared_error: 1.3419

Epoch 2/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 1.6277 - root_mean_squared_error: 1.2758
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.3461 - root_mean_squared_error: 1.1602 - val_loss: 1.2079 - val_root_mean_squared_error: 1.0991

Epoch 3/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 1.1547 - root_mean_squared_error: 1.0745
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9880 - root_mean_squared_error: 0.9940 - val_loss: 1.0114 - val_root_mean_squared_error: 1.0057

Epoch 4/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.1176 - root_mean_squa

2025/12/31 12:31:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 328ms/step - loss: 30.9881 - root_mean_squared_error: 5.5667
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 29.9603 - root_mean_squared_error: 5.4736 - val_loss: 27.5955 - val_root_mean_squared_error: 5.2531

Epoch 2/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 28.9131 - root_mean_squared_error: 5.3771
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 26.9929 - root_mean_squared_error: 5.1955 - val_loss: 24.8740 - val_root_mean_squared_error: 4.9874

Epoch 3/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 26.5810 - root_mean_squared_error: 5.1557
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 24.3130 - root_mean_squared_error: 4.9308 - val_loss: 22.4090 - val_root_mean_squared_error: 4.7338

Epoch 4/50                               

2025/12/31 12:31:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

2025/12/31 12:32:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 299ms/step - loss: 32.8911 - root_mean_squared_error: 5.7351
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 7.0543 - root_mean_squared_error: 2.6560 - val_loss: 1.5393 - val_root_mean_squared_error: 1.2407

Epoch 2/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 1.9683 - root_mean_squared_error: 1.4030
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.3361 - root_mean_squared_error: 1.1559 - val_loss: 1.1464 - val_root_mean_squared_error: 1.0707

Epoch 3/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.9680 - root_mean_squared_error: 0.9839
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.0404 - root_mean_squared_error: 1.0200 - val_loss: 0.9256 - val_root_mean_squared_error: 0.9621

Epoch 4/50                                       

2025/12/31 12:32:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:58<00:00, 14.64s/trial, best loss: 0.7071940898895264]


TypeError: log_param() missing 1 required positional argument: 'value'

In [22]:
import mlflow
import mlflow.tensorflow
from hyperopt import Trials, fmin, tpe


# Set MLflow experiment
mlflow.set_experiment("DL-MLflow-Wine-Quality")

with mlflow.start_run():

    # Conduct hyperparameter optimization
    trials = Trials()

    best_params = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Get best trial based on minimum loss
    best_trial = sorted(trials.results, key=lambda x: x["loss"])[0]

    best_loss = best_trial["loss"]
    best_model = best_trial["model"]

    # Log best hyperparameters (individually)
    for param, value in best_params.items():
        mlflow.log_param(param, value)

    # Log best metric
    mlflow.log_metric("eval_rmse", best_loss)

    # Log best model
    mlflow.tensorflow.log_model(
        best_model,
        artifact_path="model",
        signature=signature
    )

    # Print results
    print(f"Best Parameters: {best_params}")
    print(f"Best eval RMSE: {best_loss}")


Epoch 1/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 328ms/step - loss: 31.8135 - root_mean_squared_error: 5.6403
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 27.4867 - root_mean_squared_error: 5.2428 - val_loss: 22.7439 - val_root_mean_squared_error: 4.7691

Epoch 2/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 22.8145 - root_mean_squared_error: 4.7765
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 21.7697 - root_mean_squared_error: 4.6658 - val_loss: 17.9495 - val_root_mean_squared_error: 4.2367

Epoch 3/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 20.0142 - root_mean_squared_error: 4.4737
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 17.2866 - root_mean_squared_error: 4.1577 - val_loss: 14.1735 - val_root_mean_squared_error: 3.7648

Epoch 4/50                                           

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 16.2665 - root_m

2025/12/31 12:34:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/50                                                                    

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 315ms/step - loss: 37.3379 - root_mean_squared_error: 6.1105
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.2755 - root_mean_squared_error: 2.5051 - val_loss: 1.3630 - val_root_mean_squared_error: 1.1675

Epoch 2/50                                                                    

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 1.6422 - root_mean_squared_error: 1.2815
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1494 - root_mean_squared_error: 1.0721 - val_loss: 1.0525 - val_root_mean_squared_error: 1.0259

Epoch 3/50                                                                    

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 1.3580 - root_mean_squared_error: 1.1653
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8857 - root_mean_squared_error: 0.9411 - val_loss: 0.8735 - val_root_mean_squared_error: 0.9346

Epoch 4/50                                          

2025/12/31 12:34:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 296ms/step - loss: 36.8884 - root_mean_squared_error: 6.0736
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 35.0216 - root_mean_squared_error: 5.9179 - val_loss: 30.6796 - val_root_mean_squared_error: 5.5389

Epoch 2/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 35.4658 - root_mean_squared_error: 5.9553
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 30.1325 - root_mean_squared_error: 5.4893 - val_loss: 26.4186 - val_root_mean_squared_error: 5.1399

Epoch 3/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 25.8062 - root_mean_squared_error: 5.0800
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 25.9793 - root_mean_squared_error: 5.0970 - val_loss: 22.7661 - val_root_mean_squared_error: 4.7714

Epoch 4/50                               

...

Epoch 1/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 4s 308ms/step - loss: 32.3181 - root_mean_squared_error: 5.6849
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 4.0157 - root_mean_squared_error: 2.0039 - val_loss: 1.0423 - val_root_mean_squared_error: 1.0210

Epoch 2/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.1895 - root_mean_squared_error: 1.0906
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8624 - root_mean_squared_error: 0.9286 - val_loss: 0.8460 - val_root_mean_squared_error: 0.9198

Epoch 3/50                                                                     

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.7297 - root_mean_squared_error: 0.8542
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6863 - root_mean_squared_error: 0.8284 - val_loss: 0.6971 - val_root_mean_squared_error: 0.8349

Epoch 4/50                                       

2025/12/31 12:35:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:53<00:00, 13.42s/trial, best loss: 0.7131500840187073]

2025/12/31 12:35:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Parameters: {'lr': np.float64(0.030770641282547966), 'momentum': np.float64(0.0027617075273810524)}
Best eval RMSE: 0.7131500840187073


ravel - 1D array

##### train_x=train.drop('quality',axis=1).values() - exclude quality columns rest all in training set - group of independant variables

##### train_y=train['quality'].values.ravel() - only quality column - group of predict or depeandant column
